In [ ]:
# ============================================================
# AI REALIZATION GAP — EU27
# STEP 03 — TIME-VARYING AI POTENTIAL INDEX
# ============================================================

from google.colab import drive
from pathlib import Path

import requests
import pandas as pd
import numpy as np
import itertools
import json
from datetime import datetime

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AI_Realization_Gap_EU27"
)

ADMIN_DIR = PROJECT_ROOT / "00_admin"
RAW_DIR = PROJECT_ROOT / "01_raw_data"
INTERMEDIATE_DIR = PROJECT_ROOT / "02_intermediate_data"
PROCESSED_DIR = PROJECT_ROOT / "03_processed_data"
TABLE_DIR = PROJECT_ROOT / "05_tables"
LOG_DIR = PROJECT_ROOT / "08_logs"

for d in [
    ADMIN_DIR,
    RAW_DIR,
    INTERMEDIATE_DIR,
    PROCESSED_DIR,
    TABLE_DIR,
    LOG_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

AI_YEARS = [2021, 2023, 2024, 2025]

print("Project root:", PROJECT_ROOT)
print("Years:", AI_YEARS)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/AI_Realization_Gap_EU27
Years: [2021, 2023, 2024, 2025]


In [ ]:
# ============================================================
# LOAD COUNTRY MASTER
# ============================================================

country_master = pd.read_csv(
    ADMIN_DIR / "eu27_country_master.csv"
)

EU27 = country_master["geo"].tolist()

print("Countries:", len(EU27))

display(country_master.head())

Countries: 27


,geo,country,iso3
0,AT,Austria,AUT
1,BE,Belgium,BEL
2,BG,Bulgaria,BGR
3,HR,Croatia,HRV
4,CY,Cyprus,CYP


In [ ]:
# ============================================================
# EUROSTAT JSON-STAT HELPER
# ============================================================

def ordered_codes(dataset, dim):

    idx = (
        dataset["dimension"][dim]
        ["category"]["index"]
    )

    if isinstance(idx, dict):
        return [
            code
            for code, position
            in sorted(
                idx.items(),
                key=lambda x: x[1]
            )
        ]

    return idx


def eurostat_json_to_df(dataset):

    dims = dataset["id"]

    dimension_codes = {
        dim: ordered_codes(dataset, dim)
        for dim in dims
    }

    combinations = itertools.product(
        *[
            dimension_codes[d]
            for d in dims
        ]
    )

    values = dataset.get("value", {})

    records = []

    for i, combination in enumerate(combinations):

        row = dict(
            zip(dims, combination)
        )

        row["value"] = values.get(
            str(i),
            np.nan
        )

        records.append(row)

    return pd.DataFrame(records)

In [ ]:
# ============================================================
# COMPONENT 1 — ICT SPECIALISTS
# COMPLETE RESTART BLOCK
# ============================================================

import requests
import pandas as pd
import numpy as np
import itertools

ICT_DATASET = "isoc_sks_itspt"

# ------------------------------------------------------------
# 1. HELPER FUNCTIONS
# ------------------------------------------------------------

def ordered_codes(dataset, dim):

    idx = (
        dataset["dimension"][dim]
        ["category"]["index"]
    )

    if isinstance(idx, dict):
        return [
            code
            for code, position
            in sorted(
                idx.items(),
                key=lambda x: x[1]
            )
        ]

    return idx


def eurostat_json_to_df(dataset):

    dims = dataset["id"]

    dimension_codes = {
        dim: ordered_codes(dataset, dim)
        for dim in dims
    }

    combinations = itertools.product(
        *[
            dimension_codes[d]
            for d in dims
        ]
    )

    values = dataset.get("value", {})

    records = []

    for i, combination in enumerate(combinations):

        row = dict(
            zip(dims, combination)
        )

        row["value"] = values.get(
            str(i),
            np.nan
        )

        records.append(row)

    return pd.DataFrame(records)


# ------------------------------------------------------------
# 2. DOWNLOAD ICT SPECIALISTS DATA
# ------------------------------------------------------------

ict_url = (
    "https://ec.europa.eu/eurostat/api/"
    "dissemination/statistics/1.0/data/"
    f"{ICT_DATASET}"
    "?lang=en"
    "&sinceTimePeriod=2021"
    "&untilTimePeriod=2025"
)

response = requests.get(
    ict_url,
    timeout=120
)

response.raise_for_status()

ict_json = response.json()

print("Dataset downloaded.")
print("Dimensions:", ict_json["id"])
print("Sizes:", ict_json["size"])


# ------------------------------------------------------------
# 3. CONVERT TO DATAFRAME
# ------------------------------------------------------------

ict_long = eurostat_json_to_df(
    ict_json
)

print("\nict_long created successfully.")
print("Shape:", ict_long.shape)

display(ict_long.head())# ============================================================
# COMPONENT 1 — ICT SPECIALISTS
# ============================================================

ICT_DATASET = "isoc_sks_itspt"

ict_url = (
    "https://ec.europa.eu/eurostat/api/"
    "dissemination/statistics/1.0/data/"
    f"{ICT_DATASET}"
    "?lang=en"
    "&sinceTimePeriod=2021"
    "&untilTimePeriod=2025"
)

response = requests.get(
    ict_url,
    timeout=120
)

response.raise_for_status()

ict_json = response.json()

print("ICT dimensions:", ict_json["id"])
print("ICT sizes:", ict_json["size"])

Dataset downloaded.
Dimensions: ['freq', 'unit', 'geo', 'time']
Sizes: [1, 2, 37, 5]

ict_long created successfully.
Shape: (370, 5)


,freq,unit,geo,time,value
0,A,THS_PER,EU27_2020,2021,8920.3
1,A,THS_PER,EU27_2020,2022,9370.1
2,A,THS_PER,EU27_2020,2023,9749.5
3,A,THS_PER,EU27_2020,2024,10189.9
4,A,THS_PER,EU27_2020,2025,10450.8


ICT dimensions: ['freq', 'unit', 'geo', 'time']
ICT sizes: [1, 2, 37, 5]


In [ ]:
# ============================================================
# ICT SPECIALISTS — UNIT CHECK
# ============================================================

unit_labels = (
    ict_json["dimension"]["unit"]
    ["category"]
    .get("label", {})
)

print("Available units:")

for code, label in unit_labels.items():
    print(code, "->", label)

Available units:
THS_PER -> Thousand persons
PC_EMP -> Percentage of total employment


In [ ]:
# ============================================================
# FILTER ICT SPECIALISTS — PERCENT OF TOTAL EMPLOYMENT
# ============================================================

ICT_UNIT = "PC_EMP"

ict = ict_long[
    (ict_long["freq"] == "A") &
    (ict_long["unit"] == ICT_UNIT) &
    (ict_long["geo"].isin(EU27)) &
    (
        ict_long["time"]
        .astype(str)
        .isin(
            ["2021", "2023", "2024", "2025"]
        )
    )
].copy()

ict["year"] = ict["time"].astype(int)

ict["ict_specialists_pct"] = pd.to_numeric(
    ict["value"],
    errors="coerce"
)

ict = ict[
    [
        "geo",
        "year",
        "ict_specialists_pct"
    ]
].sort_values(
    ["geo", "year"]
).reset_index(drop=True)

print("ICT rows:", len(ict))
print("Countries:", ict["geo"].nunique())
print("Years:", sorted(ict["year"].unique()))

display(ict.head(20))

ICT rows: 108
Countries: 27
Years: [np.int64(2021), np.int64(2023), np.int64(2024), np.int64(2025)]


,geo,year,ict_specialists_pct
0,AT,2021,4.5
1,AT,2023,5.3
2,AT,2024,5.3
3,AT,2025,5.4
4,BE,2021,5.6
5,BE,2023,5.4
6,BE,2024,5.7
7,BE,2025,5.9
8,BG,2021,3.5
9,BG,2023,4.3


In [ ]:
# ============================================================
# ICT QUALITY CONTROL
# ============================================================

print("Missing values:")
print(
    ict["ict_specialists_pct"]
    .isna()
    .sum()
)

print("\nDuplicates:")
print(
    ict.duplicated(
        ["geo", "year"]
    ).sum()
)

print("\nDescriptive statistics:")
display(
    ict["ict_specialists_pct"]
    .describe()
)

Missing values:
0

Duplicates:
0

Descriptive statistics:


,ict_specialists_pct
count,108.000000
mean,5.128704
std,1.454227
min,2.400000
25%,4.300000
50%,4.800000
75%,5.725000
max,8.900000


In [ ]:
# ============================================================
# ICT TIME VARIATION CHECK
# ============================================================

ict_variation = (
    ict
    .groupby("geo")
    ["ict_specialists_pct"]
    .agg(
        min="min",
        max="max",
        std="std"
    )
    .reset_index()
)

ict_variation["range"] = (
    ict_variation["max"]
    - ict_variation["min"]
)

display(
    ict_variation.sort_values(
        "range",
        ascending=False
    )
)

print(
    "Countries with changing ICT specialist share:",
    (ict_variation["range"] > 0).sum()
)

,geo,min,max,std,range
17,LU,6.7,8.7,0.834666,2.0
16,LT,3.8,5.7,0.818026,1.9
3,CY,3.9,5.6,0.704746,1.7
2,BG,3.5,4.8,0.571548,1.3
12,HR,3.7,5.0,0.602080,1.3
18,LV,3.8,4.9,0.454606,1.1
25,SI,3.8,4.8,0.420317,1.0
7,EE,6.2,7.2,0.411299,1.0
0,AT,4.5,5.4,0.419325,0.9
24,SE,8.0,8.9,0.387298,0.9


Countries with changing ICT specialist share: 27


In [ ]:
# ============================================================
# SAVE CLEAN ICT COMPONENT
# ============================================================

ict_output = (
    INTERMEDIATE_DIR /
    "Step_03_ICT_specialists_EU27.csv"
)

ict.to_csv(
    ict_output,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(ict_output)

Saved:
/content/drive/MyDrive/AI_Realization_Gap_EU27/02_intermediate_data/Step_03_ICT_specialists_EU27.csv


In [ ]:
# ============================================================
# COMPONENT 2 — HUMAN RESOURCES IN SCIENCE & TECHNOLOGY
# ============================================================

HRST_DATASET = "hrst_st_ncat"

hrst_url = (
    "https://ec.europa.eu/eurostat/api/"
    "dissemination/statistics/1.0/data/"
    f"{HRST_DATASET}"
    "?lang=en"
    "&sinceTimePeriod=2021"
    "&untilTimePeriod=2025"
)

response = requests.get(
    hrst_url,
    timeout=180
)

response.raise_for_status()

hrst_json = response.json()

print("HRST dimensions:", hrst_json["id"])
print("HRST sizes:", hrst_json["size"])

HRST dimensions: ['freq', 'unit', 'category', 'age', 'sex', 'geo', 'time']
HRST sizes: [1, 3, 5, 6, 3, 38, 5]


In [ ]:
hrst_long = eurostat_json_to_df(
    hrst_json
)

print("HRST long shape:", hrst_long.shape)

display(hrst_long.head())

HRST long shape: (51300, 8)


,freq,unit,category,age,sex,geo,time,value
0,A,THS_PER,HRST,Y15-24_Y65-74,T,EU27_2020,2021,17285.1
1,A,THS_PER,HRST,Y15-24_Y65-74,T,EU27_2020,2022,17818.4
2,A,THS_PER,HRST,Y15-24_Y65-74,T,EU27_2020,2023,18491.7
3,A,THS_PER,HRST,Y15-24_Y65-74,T,EU27_2020,2024,19027.2
4,A,THS_PER,HRST,Y15-24_Y65-74,T,EU27_2020,2025,19687.6


In [ ]:
for variable in hrst_json["id"]:

    print("\n" + "=" * 70)
    print(variable)

    labels = (
        hrst_json["dimension"][variable]
        ["category"]
        .get("label", {})
    )

    for code, label in labels.items():
        print(code, "->", label)


freq
A -> Annual

unit
THS_PER -> Thousand persons
PC_POP -> Percentage of total population
PC_ACT -> Percentage of population in the labour force

category
HRST -> Persons with tertiary education (ISCED) and/or employed in science and technology
HRSTE -> Persons with tertiary education (ISCED)
HRSTO -> Persons employed in science and technology
HRSTC -> Persons with tertiary education (ISCED) and employed in science and technology
SE -> Scientists and engineers

age
Y15-24_Y65-74 -> From 15 to 24 years and 65 to 74 years
Y15-74 -> From 15 to 74 years
Y25-34 -> From 25 to 34 years
Y25-64 -> From 25 to 64 years
Y35-44 -> From 35 to 44 years
Y45-64 -> From 45 to 64 years

sex
T -> Total
M -> Males
F -> Females

geo
EU27_2020 -> European Union - 27 countries (from 2020)
EA21 -> Euro area – 21 countries (from 2026)
BE -> Belgium
BG -> Bulgaria
CZ -> Czechia
DK -> Denmark
DE -> Germany
EE -> Estonia
IE -> Ireland
EL -> Greece
ES -> Spain
FR -> France
HR -> Croatia
IT -> Italy
CY -> Cyprus


In [ ]:
# ============================================================
# FILTER HRST
# ============================================================

HRST_CATEGORY = "HRST"
HRST_AGE = "Y25-64"
HRST_SEX = "T"
HRST_UNIT = "PC_ACT"

hrst = hrst_long[
    (hrst_long["freq"] == "A") &
    (hrst_long["unit"] == HRST_UNIT) &
    (hrst_long["category"] == HRST_CATEGORY) &
    (hrst_long["age"] == HRST_AGE) &
    (hrst_long["sex"] == HRST_SEX) &
    (hrst_long["geo"].isin(EU27)) &
    (
        hrst_long["time"]
        .astype(str)
        .isin(["2021", "2023", "2024", "2025"])
    )
].copy()

hrst["year"] = hrst["time"].astype(int)

hrst["hrst_pct"] = pd.to_numeric(
    hrst["value"],
    errors="coerce"
)

hrst = hrst[
    [
        "geo",
        "year",
        "hrst_pct"
    ]
].sort_values(
    ["geo", "year"]
).reset_index(drop=True)

print("HRST rows:", len(hrst))
print("Countries:", hrst["geo"].nunique())
print("Years:", sorted(hrst["year"].unique()))

display(hrst.head(20))

HRST rows: 108
Countries: 27
Years: [np.int64(2021), np.int64(2023), np.int64(2024), np.int64(2025)]


,geo,year,hrst_pct
0,AT,2021,52.3
1,AT,2023,54.5
2,AT,2024,55.2
3,AT,2025,55.3
4,BE,2021,58.7
5,BE,2023,58.5
6,BE,2024,59.0
7,BE,2025,61.7
8,BG,2021,38.5
9,BG,2023,39.0


In [ ]:
# ============================================================
# HRST QUALITY CONTROL
# ============================================================

print("Missing values:")
print(
    hrst["hrst_pct"]
    .isna()
    .sum()
)

print("\nDuplicates:")
print(
    hrst.duplicated(
        ["geo", "year"]
    ).sum()
)

print("\nDescriptive statistics:")

display(
    hrst["hrst_pct"]
    .describe()
)

Missing values:
0

Duplicates:
0

Descriptive statistics:


,hrst_pct
count,108.000000
mean,51.701852
std,9.471048
min,29.900000
25%,43.875000
50%,52.200000
75%,58.550000
max,73.000000


In [ ]:
# ============================================================
# HRST TIME VARIATION
# ============================================================

hrst_variation = (
    hrst
    .groupby("geo")
    ["hrst_pct"]
    .agg(
        min="min",
        max="max",
        std="std"
    )
    .reset_index()
)

hrst_variation["range"] = (
    hrst_variation["max"]
    - hrst_variation["min"]
)

display(
    hrst_variation.sort_values(
        "range",
        ascending=False
    )
)

print(
    "Countries with changing HRST:",
    (hrst_variation["range"] > 0).sum()
)

,geo,min,max,std,range
2,BG,38.5,46.4,3.710683,7.9
25,SI,48.3,55.8,3.491299,7.5
12,HR,39.2,44.8,2.503830,5.6
16,LT,54.2,59.7,2.291106,5.5
21,PL,47.5,52.6,2.359908,5.1
17,LU,68.1,73.0,2.551307,4.9
3,CY,54.8,59.6,2.128184,4.8
14,IE,63.0,67.6,2.032035,4.6
11,FR,55.0,58.8,1.590335,3.8
20,NL,62.6,66.3,1.558846,3.7


Countries with changing HRST: 27


In [ ]:
# ============================================================
# SAVE CLEAN HRST COMPONENT
# ============================================================

hrst_output = (
    INTERMEDIATE_DIR /
    "Step_03_HRST_EU27.csv"
)

hrst.to_csv(
    hrst_output,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(hrst_output)

Saved:
/content/drive/MyDrive/AI_Realization_Gap_EU27/02_intermediate_data/Step_03_HRST_EU27.csv


In [ ]:
# ============================================================
# COMPONENT 3 — HIGH-TECH EMPLOYMENT
# ============================================================

HT_DATASET = "htec_emp_nat2"

ht_url = (
    "https://ec.europa.eu/eurostat/api/"
    "dissemination/statistics/1.0/data/"
    f"{HT_DATASET}"
    "?lang=en"
    "&sinceTimePeriod=2021"
    "&untilTimePeriod=2025"
)

response = requests.get(
    ht_url,
    timeout=180
)

response.raise_for_status()

ht_json = response.json()

print("High-tech dimensions:", ht_json["id"])
print("High-tech sizes:", ht_json["size"])

High-tech dimensions: ['freq', 'sex', 'nace_r2', 'unit', 'geo', 'time']
High-tech sizes: [1, 3, 33, 2, 38, 5]


In [ ]:
# ============================================================
# CONVERT HIGH-TECH DATA TO LONG FORMAT
# ============================================================

ht_long = eurostat_json_to_df(
    ht_json
)

print("High-tech long shape:", ht_long.shape)

display(ht_long.head())

High-tech long shape: (37620, 7)


,freq,sex,nace_r2,unit,geo,time,value
0,A,T,TOTAL,THS_PER,EU27_2020,2021,197198.9
1,A,T,TOTAL,THS_PER,EU27_2020,2022,201977.2
2,A,T,TOTAL,THS_PER,EU27_2020,2023,204429.0
3,A,T,TOTAL,THS_PER,EU27_2020,2024,206449.6
4,A,T,TOTAL,THS_PER,EU27_2020,2025,207841.9


In [ ]:
# ============================================================
# SAVE RAW HIGH-TECH DATA
# ============================================================

ht_raw_path = (
    RAW_DIR /
    "Step_03_HighTechEmployment_raw.json"
)

with open(
    ht_raw_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ht_json,
        f,
        ensure_ascii=False
    )

print("Saved:")
print(ht_raw_path)

Saved:
/content/drive/MyDrive/AI_Realization_Gap_EU27/01_raw_data/Step_03_HighTechEmployment_raw.json


In [ ]:
# ============================================================
# INSPECT HIGH-TECH DIMENSIONS
# ============================================================

for variable in ht_json["id"]:

    print("\n" + "=" * 70)
    print(variable)

    labels = (
        ht_json["dimension"][variable]
        ["category"]
        .get("label", {})
    )

    for code, label in labels.items():

        print(
            code,
            "->",
            label
        )


freq
A -> Annual

sex
T -> Total
M -> Males
F -> Females

nace_r2
TOTAL -> Total - all NACE activities
A_B -> Agriculture, forestry and fishing; mining and quarrying
HTC -> High-technology sectors (high-technology manufacturing and knowledge-intensive high-technology services)
C -> Manufacturing
C_HTC_MH -> High and medium high-technology manufacturing
C_HTC_M -> Medium high-technology manufacturing
C_HTC -> High-technology manufacturing
C_LTC_LM -> Low and medium low-technology manufacturing
C_LTC_M -> Medium low-technology manufacturing
C_LTC -> Low-technology manufacturing
D-F -> Electricity, gas, steam and air conditioning supply; water supply and construction
G-U -> Services
G_I_T -> Wholesale and retail trade; accommodation and food service activities; activities of households as employers
H49-H52_N79 -> Land transport, transport via pipelines, water transport, air transport, warehousing and support activities for transportation; travel agency, tour operator and other reservatio

In [ ]:
# ============================================================
# FILTER HIGH-TECH EMPLOYMENT
# ============================================================

HT_NACE = "HTC"
HT_UNIT = "PC_EMP"
HT_SEX = "T"

ht = ht_long[
    (ht_long["freq"] == "A") &
    (ht_long["sex"] == HT_SEX) &
    (ht_long["nace_r2"] == HT_NACE) &
    (ht_long["unit"] == HT_UNIT) &
    (ht_long["geo"].isin(EU27)) &
    (
        ht_long["time"]
        .astype(str)
        .isin(["2021", "2023", "2024", "2025"])
    )
].copy()

ht["year"] = ht["time"].astype(int)

ht["hightech_emp_pct"] = pd.to_numeric(
    ht["value"],
    errors="coerce"
)

ht = ht[
    [
        "geo",
        "year",
        "hightech_emp_pct"
    ]
].sort_values(
    ["geo", "year"]
).reset_index(drop=True)

print("High-tech rows:", len(ht))
print("Countries:", ht["geo"].nunique())
print("Years:", sorted(ht["year"].unique()))

display(ht.head(20))

High-tech rows: 108
Countries: 27
Years: [np.int64(2021), np.int64(2023), np.int64(2024), np.int64(2025)]


,geo,year,hightech_emp_pct
0,AT,2021,4.5
1,AT,2023,5.1
2,AT,2024,5.2
3,AT,2025,5.4
4,BE,2021,5.7
5,BE,2023,5.5
6,BE,2024,5.6
7,BE,2025,5.9
8,BG,2021,4.5
9,BG,2023,5.1


In [ ]:
# ============================================================
# HIGH-TECH EMPLOYMENT QUALITY CONTROL
# ============================================================

print("Missing values:")
print(
    ht["hightech_emp_pct"]
    .isna()
    .sum()
)

print("\nDuplicates:")
print(
    ht.duplicated(
        ["geo", "year"]
    ).sum()
)

print("\nDescriptive statistics:")

display(
    ht["hightech_emp_pct"]
    .describe()
)

Missing values:
0

Duplicates:
0

Descriptive statistics:


,hightech_emp_pct
count,108.000000
mean,5.457407
std,1.323208
min,3.200000
25%,4.675000
50%,5.400000
75%,6.200000
max,10.200000


In [ ]:
# ============================================================
# HIGH-TECH EMPLOYMENT QUALITY CONTROL
# ============================================================

print("Missing values:")
print(
    ht["hightech_emp_pct"]
    .isna()
    .sum()
)

print("\nDuplicates:")
print(
    ht.duplicated(
        ["geo", "year"]
    ).sum()
)

print("\nDescriptive statistics:")

display(
    ht["hightech_emp_pct"]
    .describe()
)

Missing values:
0

Duplicates:
0

Descriptive statistics:


,hightech_emp_pct
count,108.000000
mean,5.457407
std,1.323208
min,3.200000
25%,4.675000
50%,5.400000
75%,6.200000
max,10.200000


In [ ]:
# ============================================================
# HIGH-TECH EMPLOYMENT TIME VARIATION
# ============================================================

ht_variation = (
    ht
    .groupby("geo")
    ["hightech_emp_pct"]
    .agg(
        min="min",
        max="max",
        std="std"
    )
    .reset_index()
)

ht_variation["range"] = (
    ht_variation["max"]
    - ht_variation["min"]
)

display(
    ht_variation.sort_values(
        "range",
        ascending=False
    )
)

print(
    "Countries with changing high-tech employment share:",
    (ht_variation["range"] > 0).sum()
)

,geo,min,max,std,range
3,CY,3.9,6.3,0.997914,2.4
12,HR,3.6,5.4,0.826640,1.8
16,LT,4.3,5.7,0.632456,1.4
25,SI,5.6,6.8,0.499166,1.2
21,PL,3.7,4.8,0.499166,1.1
18,LV,4.5,5.5,0.434933,1.0
22,PT,4.1,5.0,0.403113,0.9
17,LU,5.0,5.9,0.377492,0.9
19,MT,5.6,6.5,0.424264,0.9
0,AT,4.5,5.4,0.387298,0.9


Countries with changing high-tech employment share: 27


In [ ]:
# ============================================================
# SAVE CLEAN HIGH-TECH COMPONENT
# ============================================================

ht_output = (
    INTERMEDIATE_DIR /
    "Step_03_HighTechEmployment_EU27.csv"
)

ht.to_csv(
    ht_output,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(ht_output)

Saved:
/content/drive/MyDrive/AI_Realization_Gap_EU27/02_intermediate_data/Step_03_HighTechEmployment_EU27.csv


In [ ]:
# ============================================================
# MERGE ALL AI POTENTIAL COMPONENTS
# ============================================================

skeleton = pd.read_csv(
    INTERMEDIATE_DIR /
    "eu27_ai_year_panel_skeleton.csv"
)

potential = (
    skeleton
    .merge(
        ict,
        on=["geo", "year"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        hrst,
        on=["geo", "year"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        ht,
        on=["geo", "year"],
        how="left",
        validate="one_to_one"
    )
)

print("Panel shape:", potential.shape)

display(potential.head(15))

Panel shape: (108, 7)


,geo,country,iso3,year,ict_specialists_pct,hrst_pct,hightech_emp_pct
0,AT,Austria,AUT,2021,4.5,52.3,4.5
1,AT,Austria,AUT,2023,5.3,54.5,5.1
2,AT,Austria,AUT,2024,5.3,55.2,5.2
3,AT,Austria,AUT,2025,5.4,55.3,5.4
4,BE,Belgium,BEL,2021,5.6,58.7,5.7
5,BE,Belgium,BEL,2023,5.4,58.5,5.5
6,BE,Belgium,BEL,2024,5.7,59.0,5.6
7,BE,Belgium,BEL,2025,5.9,61.7,5.9
8,BG,Bulgaria,BGR,2021,3.5,38.5,4.5
9,BG,Bulgaria,BGR,2023,4.3,39.0,5.1


In [ ]:
# ============================================================
# COMPLETE-CASE CHECK
# ============================================================

variables = [
    "ict_specialists_pct",
    "hrst_pct",
    "hightech_emp_pct"
]

print("Missing values by variable:")

display(
    potential[variables]
    .isna()
    .sum()
    .to_frame("missing_n")
)

print(
    "\nRows with any missing:",
    potential[variables]
    .isna()
    .any(axis=1)
    .sum()
)

Missing values by variable:


,missing_n
ict_specialists_pct,0
hrst_pct,0
hightech_emp_pct,0



Rows with any missing: 0


In [ ]:
# ============================================================
# COMPONENT CORRELATIONS
# ============================================================

component_corr = potential[
    [
        "ict_specialists_pct",
        "hrst_pct",
        "hightech_emp_pct"
    ]
].corr()

display(component_corr)

,ict_specialists_pct,hrst_pct,hightech_emp_pct
ict_specialists_pct,1.000000,0.807451,0.612849
hrst_pct,0.807451,1.000000,0.566787
hightech_emp_pct,0.612849,0.566787,1.000000


In [ ]:
# ============================================================
# GLOBAL Z-SCORE STANDARDIZATION
# ============================================================

components = [
    "ict_specialists_pct",
    "hrst_pct",
    "hightech_emp_pct"
]

for var in components:

    mean_value = potential[var].mean()
    std_value = potential[var].std(ddof=0)

    potential[f"z_{var}"] = (
        potential[var] - mean_value
    ) / std_value

In [ ]:
# ============================================================
# PCA-BASED AI POTENTIAL INDEX
# ============================================================

from sklearn.decomposition import PCA

pca_vars = [
    "z_ict_specialists_pct",
    "z_hrst_pct",
    "z_hightech_emp_pct"
]

X = potential[pca_vars].copy()

pca = PCA(n_components=3)

pca_scores = pca.fit_transform(X)

print("Explained variance ratios:")
print(pca.explained_variance_ratio_)

Explained variance ratios:
[0.77685381 0.15989348 0.06325271]


In [ ]:
# ============================================================
# PCA LOADINGS
# ============================================================

loadings = pd.DataFrame(
    {
        "variable": [
            "ICT specialists",
            "HRST",
            "High-tech employment"
        ],
        "PC1_loading": pca.components_[0]
    }
)

display(loadings)

,variable,PC1_loading
0,ICT specialists,0.604754
1,HRST,0.593308
2,High-tech employment,0.531281


In [ ]:
# ============================================================
# PCA-DERIVED WEIGHTS
# ============================================================

pc1_loadings = np.array([
    0.604754,
    0.593308,
    0.531281
])

weights = (
    pc1_loadings /
    pc1_loadings.sum()
)

w_ict, w_hrst, w_ht = weights

print("ICT weight:", w_ict)
print("HRST weight:", w_hrst)
print("High-tech weight:", w_ht)
print("Sum:", weights.sum())

ICT weight: 0.3497015918762212
HRST weight: 0.34308289333001024
High-tech weight: 0.30721551479376846
Sum: 1.0


In [ ]:
# ============================================================
# PCA-WEIGHTED AI POTENTIAL
# ============================================================

potential["ai_potential_pca_z"] = (
    w_ict
    * potential["z_ict_specialists_pct"]
    +
    w_hrst
    * potential["z_hrst_pct"]
    +
    w_ht
    * potential["z_hightech_emp_pct"]
)

potential[
    [
        "country",
        "year",
        "ai_potential_pca_z"
    ]
].head(20)

,country,year,ai_potential_pca_z
0,Austria,2021,-0.353444
1,Austria,2023,0.059850
2,Austria,2024,0.108651
3,Austria,2025,0.183101
4,Belgium,2021,0.425134
5,Belgium,2023,0.322885
6,Belgium,2024,0.436886
7,Belgium,2025,0.653443
8,Bulgaria,2021,-1.097265
9,Bulgaria,2023,-0.745839


In [ ]:
# ============================================================
# NORMALIZE PCA INDEX TO 0-1
# ============================================================

pca_min = potential[
    "ai_potential_pca_z"
].min()

pca_max = potential[
    "ai_potential_pca_z"
].max()

potential["ai_potential"] = (
    potential["ai_potential_pca_z"]
    - pca_min
) / (
    pca_max - pca_min
)

potential[
    [
        "country",
        "year",
        "ai_potential_pca_z",
        "ai_potential"
    ]
].head(20)

,country,year,ai_potential_pca_z,ai_potential
0,Austria,2021,-0.353444,0.412078
1,Austria,2023,0.059850,0.520042
2,Austria,2024,0.108651,0.532790
3,Austria,2025,0.183101,0.552239
4,Belgium,2021,0.425134,0.615464
5,Belgium,2023,0.322885,0.588754
6,Belgium,2024,0.436886,0.618534
7,Belgium,2025,0.653443,0.675105
8,Bulgaria,2021,-1.097265,0.217772
9,Bulgaria,2023,-0.745839,0.309574


In [ ]:
# ============================================================
# EQUAL-WEIGHT AI POTENTIAL INDEX
# ROBUSTNESS VERSION
# ============================================================

potential["ai_potential_equal_z"] = (
    potential[
        [
            "z_ict_specialists_pct",
            "z_hrst_pct",
            "z_hightech_emp_pct"
        ]
    ].mean(axis=1)
)

print("Equal-weight index created.")

display(
    potential[
        [
            "country",
            "year",
            "ai_potential_equal_z"
        ]
    ].head(20)
)

Equal-weight index created.


,country,year,ai_potential_equal_z
0,Austria,2021,-0.365939
1,Austria,2023,0.047932
2,Austria,2024,0.097992
3,Austria,2025,0.175174
4,Belgium,2021,0.417378
5,Belgium,2023,0.313631
6,Belgium,2024,0.425705
7,Belgium,2025,0.643158
8,Bulgaria,2021,-1.084180
9,Bulgaria,2023,-0.730419


In [ ]:
# ============================================================
# NORMALIZE EQUAL-WEIGHT INDEX TO 0-1
# ============================================================

equal_min = potential[
    "ai_potential_equal_z"
].min()

equal_max = potential[
    "ai_potential_equal_z"
].max()

potential["ai_potential_equal"] = (
    potential["ai_potential_equal_z"] - equal_min
) / (
    equal_max - equal_min
)

display(
    potential[
        [
            "country",
            "year",
            "ai_potential_equal_z",
            "ai_potential_equal"
        ]
    ].head(20)
)

,country,year,ai_potential_equal_z,ai_potential_equal
0,Austria,2021,-0.365939,0.401139
1,Austria,2023,0.047932,0.507658
2,Austria,2024,0.097992,0.520542
3,Austria,2025,0.175174,0.540407
4,Belgium,2021,0.417378,0.602743
5,Belgium,2023,0.313631,0.576042
6,Belgium,2024,0.425705,0.604886
7,Belgium,2025,0.643158,0.660853
8,Bulgaria,2021,-1.084180,0.216285
9,Bulgaria,2023,-0.730419,0.307333


In [ ]:
# ============================================================
# PCA VS EQUAL-WEIGHT ROBUSTNESS CHECK
# ============================================================

comparison_corr = potential[
    [
        "ai_potential",
        "ai_potential_equal"
    ]
].corr()

display(comparison_corr)

,ai_potential,ai_potential_equal
ai_potential,1.000000,0.999689
ai_potential_equal,0.999689,1.000000


In [ ]:
potential["potential_index_difference"] = (
    potential["ai_potential"]
    - potential["ai_potential_equal"]
)

display(
    potential[
        [
            "country",
            "year",
            "ai_potential",
            "ai_potential_equal",
            "potential_index_difference"
        ]
    ].head(20)
)

,country,year,ai_potential,ai_potential_equal,potential_index_difference
0,Austria,2021,0.412078,0.401139,0.010939
1,Austria,2023,0.520042,0.507658,0.012384
2,Austria,2024,0.532790,0.520542,0.012248
3,Austria,2025,0.552239,0.540407,0.011832
4,Belgium,2021,0.615464,0.602743,0.012721
5,Belgium,2023,0.588754,0.576042,0.012712
6,Belgium,2024,0.618534,0.604886,0.013648
7,Belgium,2025,0.675105,0.660853,0.014252
8,Bulgaria,2021,0.217772,0.216285,0.001488
9,Bulgaria,2023,0.309574,0.307333,0.002242


In [ ]:
final_cols = [
    "geo",
    "country",
    "iso3",
    "year",
    "ict_specialists_pct",
    "hrst_pct",
    "hightech_emp_pct",
    "z_ict_specialists_pct",
    "z_hrst_pct",
    "z_hightech_emp_pct",
    "ai_potential_pca_z",
    "ai_potential",
    "ai_potential_equal_z",
    "ai_potential_equal"
]

ai_potential_panel = (
    potential[final_cols]
    .sort_values(
        ["iso3", "year"]
    )
    .reset_index(drop=True)
)

display(ai_potential_panel.head())

,geo,country,iso3,year,ict_specialists_pct,hrst_pct,hightech_emp_pct,z_ict_specialists_pct,z_hrst_pct,z_hightech_emp_pct,ai_potential_pca_z,ai_potential,ai_potential_equal_z,ai_potential_equal
0,AT,Austria,AUT,2021,4.5,52.3,4.5,-0.434344,0.063450,-0.726923,-0.353444,0.412078,-0.365939,0.401139
1,AT,Austria,AUT,2023,5.3,54.5,5.1,0.118341,0.296820,-0.271366,0.059850,0.520042,0.047932,0.507658
2,AT,Austria,AUT,2024,5.3,55.2,5.2,0.118341,0.371074,-0.195440,0.108651,0.532790,0.097992,0.520542
3,AT,Austria,AUT,2025,5.4,55.3,5.4,0.187427,0.381681,-0.043587,0.183101,0.552239,0.175174,0.540407
4,BE,Belgium,BEL,2021,5.6,58.7,5.7,0.325598,0.742344,0.184191,0.425134,0.615464,0.417378,0.602743


In [ ]:
main_output = (
    PROCESSED_DIR /
    "Step_03_AI_potential_EU27_2021_2023_2024_2025.csv"
)

ai_potential_panel.to_csv(
    main_output,
    index=False,
    encoding="utf-8-sig"
)

print("Updated file saved:")
print(main_output)

Updated file saved:
/content/drive/MyDrive/AI_Realization_Gap_EU27/03_processed_data/Step_03_AI_potential_EU27_2021_2023_2024_2025.csv


In [ ]:
# ============================================================
# DOWNLOAD STEP 03 OUTPUTS AS ZIP
# ============================================================

from pathlib import Path
import zipfile
from google.colab import files

step03_files = [
    PROJECT_ROOT / "02_intermediate_data" / "Step_03_ICT_specialists_EU27.csv",
    PROJECT_ROOT / "02_intermediate_data" / "Step_03_HRST_EU27.csv",
    PROJECT_ROOT / "02_intermediate_data" / "Step_03_HighTechEmployment_EU27.csv",
    PROJECT_ROOT / "03_processed_data" / "Step_03_AI_potential_EU27_2021_2023_2024_2025.csv",
    PROJECT_ROOT / "05_tables" / "Step_03_AI_potential_year_summary.csv",
    PROJECT_ROOT / "05_tables" / "Step_03_AI_potential_component_correlations.csv",
    PROJECT_ROOT / "08_logs" / "Step_03_QC.csv"
]

zip_path = Path("/content/Step_03_outputs.zip")

# Dosyaların varlığını kontrol et
print("Files:")

for f in step03_files:
    print(f.name, "->", f.exists())

# ZIP oluştur
with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    for f in step03_files:

        if f.exists():
            z.write(
                f,
                arcname=f.name
            )

print("\nZIP created:")
print(zip_path)

# Bilgisayara indir
files.download(str(zip_path))

Files:
Step_03_ICT_specialists_EU27.csv -> True
Step_03_HRST_EU27.csv -> True
Step_03_HighTechEmployment_EU27.csv -> True
Step_03_AI_potential_EU27_2021_2023_2024_2025.csv -> True
Step_03_AI_potential_year_summary.csv -> False
Step_03_AI_potential_component_correlations.csv -> False
Step_03_QC.csv -> False

ZIP created:
/content/Step_03_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>